[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-03-logging-artifacts.ipynb#scrollTo=aa1bb2cc)

---
# Day 3 · Logging Artifacts and Datasets
**certified-journeys / mlflow-certified** · Day 3 · Artifacts & Models

> **Goal for today:** Log a trained sklearn model, a Matplotlib figure, a Pandas DataFrame, and an entire directory as artifacts, then navigate the artifact browser to confirm everything is stored correctly.

In [ ]:
%pip install -q mlflow scikit-learn pandas matplotlib numpy

## Step 1 · Logging a Trained Model with `mlflow.sklearn.log_model`

MLflow provides **flavour-specific model loggers** for the most popular ML frameworks:

| Flavour | Function |
|---|---|
| scikit-learn | `mlflow.sklearn.log_model(model, artifact_path)` |
| PyTorch | `mlflow.pytorch.log_model(model, artifact_path)` |
| TensorFlow/Keras | `mlflow.tensorflow.log_model(model, artifact_path)` |
| XGBoost | `mlflow.xgboost.log_model(model, artifact_path)` |
| Generic Python | `mlflow.pyfunc.log_model(...)` |

All of them store the model in the **MLflow Model format**: a directory containing the serialised model plus a `MLmodel` YAML file describing flavours and the required environment (`conda.yaml` / `requirements.txt`).

This means you can reload the model later with **`mlflow.sklearn.load_model(uri)`** or the generic **`mlflow.pyfunc.load_model(uri)`** without knowing which framework was used.

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-artifacts")

with mlflow.start_run(run_name="rf-with-model") as run:
    params = {"n_estimators": 50, "max_depth": 4, "random_state": 42}
    mlflow.log_params(params)

    rf = RandomForestClassifier(**params)
    rf.fit(X_train, y_train)

    acc = accuracy_score(y_test, rf.predict(X_test))
    mlflow.log_metric("test_accuracy", acc)

    # Log the trained model as a first-class MLflow artifact
    # artifact_path = subdirectory name inside the run's artifact root
    model_info = mlflow.sklearn.log_model(
        sk_model=rf,
        artifact_path="random-forest-model",
    )

    print(f"Run ID        : {run.info.run_id[:8]}")
    print(f"Test accuracy : {acc:.4f}")
    print(f"Model URI     : {model_info.model_uri}")

# Reload the model from the logged URI — no original object needed
reloaded_model = mlflow.sklearn.load_model(model_info.model_uri)
reload_acc = accuracy_score(y_test, reloaded_model.predict(X_test))
print(f"Reloaded model test accuracy: {reload_acc:.4f}  (should match above)")

### What just happened?
- **`mlflow.sklearn.log_model`** serialises the estimator with `pickle`, writes a `MLmodel` manifest, and generates `conda.yaml` and `requirements.txt` for reproducibility.
- **`model_info.model_uri`** is a `runs:/<run_id>/random-forest-model` URI — portable across machines that share the same tracking store.
- **`mlflow.sklearn.load_model(uri)`** reconstructs the exact estimator from the stored files — the round-trip accuracy confirms the model was saved correctly.
- In the Artifacts browser, expand `random-forest-model/` to see `MLmodel`, `model.pkl`, `conda.yaml`, and `requirements.txt`.

## Step 2 · Logging a Matplotlib Figure with `mlflow.log_figure`

Instead of saving a plot to disk and calling `log_artifact`, you can pass the figure object directly to **`mlflow.log_figure`**:

```python
mlflow.log_figure(fig, "plots/my_plot.png")
```

The second argument is the **artifact path including the filename** — unlike `log_artifact` which takes a local path and an optional directory.

| Function | When to use |
|---|---|
| `log_figure(fig, path)` | In-memory matplotlib / PIL figure — no temp file needed |
| `log_artifact(local_path)` | Already-saved file on disk |
| `log_artifacts(local_dir)` | Entire directory tree |

MLflow supports both Matplotlib and Plotly figures via `log_figure`.

In [ ]:
import mlflow
import mlflow.sklearn
import matplotlib
matplotlib.use('Agg')   # headless rendering — required in scripts and CI
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-artifacts")

with mlflow.start_run(run_name="rf-with-figures") as run:
    rf = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    rf.fit(X_train, y_train)
    acc = accuracy_score(y_test, rf.predict(X_test))

    mlflow.log_params({"n_estimators": 50, "max_depth": 4})
    mlflow.log_metric("test_accuracy", acc)

    # --- Figure 1: Confusion Matrix ---
    fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        rf, X_test, y_test,
        display_labels=iris.target_names,
        ax=ax_cm
    )
    ax_cm.set_title("Confusion Matrix — Random Forest")
    fig_cm.tight_layout()
    # Pass the figure object directly — no temp file needed
    mlflow.log_figure(fig_cm, "plots/confusion_matrix.png")
    plt.close(fig_cm)

    # --- Figure 2: Feature Importance Bar Chart ---
    fig_fi, ax_fi = plt.subplots(figsize=(6, 3))
    importances = rf.feature_importances_
    indices = np.argsort(importances)[::-1]
    ax_fi.bar(range(len(importances)), importances[indices], color="steelblue")
    ax_fi.set_xticks(range(len(importances)))
    ax_fi.set_xticklabels(
        [iris.feature_names[i] for i in indices], rotation=30, ha="right"
    )
    ax_fi.set_title("Feature Importances")
    ax_fi.set_ylabel("Importance")
    fig_fi.tight_layout()
    mlflow.log_figure(fig_fi, "plots/feature_importance.png")
    plt.close(fig_fi)

    print(f"Run ID    : {run.info.run_id[:8]}")
    print(f"Test acc  : {acc:.4f}")
    print("Figures logged: plots/confusion_matrix.png, plots/feature_importance.png")

### What just happened?
- **`mlflow.log_figure(fig, artifact_path)`** handles the save-to-bytes-and-upload step internally — cleaner than saving to `/tmp` manually.
- Both figures land under `plots/` in the artifact browser — MLflow renders PNG images inline in the UI.
- **`plt.close(fig)`** after logging frees the figure from memory — important in loops that generate many plots.
- The `Agg` backend prevents a display error when running headlessly (scripts, Colab, CI).

## Step 3 · Logging a DataFrame with `mlflow.log_table`

`mlflow.log_table` logs a Pandas DataFrame (or a plain dict) as a JSON file in the artifact store. In MLflow 2.x+, logged tables appear in a special **Tables** tab in the UI with column sorting and filtering.

```python
mlflow.log_table(data=df, artifact_file="tables/predictions.json")
```

For plain CSV export (simpler, more portable), use `log_artifact` with a CSV file:

```python
df.to_csv("/tmp/predictions.csv", index=False)
mlflow.log_artifact("/tmp/predictions.csv", artifact_path="tables")
```

Use `log_table` when you want the interactive UI; use CSV when you need the file to be parseable by downstream tools.

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-artifacts")

with mlflow.start_run(run_name="rf-with-tables") as run:
    rf = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    rf.fit(X_train, y_train)

    y_pred  = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)  # shape (n_samples, 3)

    # Build a predictions DataFrame
    pred_df = pd.DataFrame({
        "true_label"  : [iris.target_names[i] for i in y_test],
        "pred_label"  : [iris.target_names[i] for i in y_pred],
        "correct"     : y_test == y_pred,
        "prob_setosa" : np.round(y_proba[:, 0], 4),
        "prob_versicolor": np.round(y_proba[:, 1], 4),
        "prob_virginica": np.round(y_proba[:, 2], 4),
    })

    # log_table: stores as JSON; renders in MLflow UI Tables tab
    mlflow.log_table(data=pred_df, artifact_file="tables/predictions.json")

    # Also log as CSV for downstream portability
    csv_path = "/tmp/predictions.csv"
    pred_df.to_csv(csv_path, index=False)
    mlflow.log_artifact(csv_path, artifact_path="tables")

    acc = accuracy_score(y_test, y_pred)
    mlflow.log_params({"n_estimators": 50, "max_depth": 4})
    mlflow.log_metric("test_accuracy", acc)

    print(f"Run ID       : {run.info.run_id[:8]}")
    print(f"Test accuracy: {acc:.4f}")
    print(f"Rows in pred table: {len(pred_df)}")
    print("\nSample (first 5 rows):")
    print(pred_df.head().to_string(index=False))

### What just happened?
- **`log_table`** converts the DataFrame to JSON and stores it under the `tables/` subdirectory.
- In MLflow 2.x+, the Tables tab in the UI renders this JSON with sortable columns — great for auditing predictions.
- **Logging both formats** (JSON for UI + CSV for tools) is a practical compromise: keep the JSON for the UI and the CSV for downstream pipelines.
- The `correct` boolean column makes it easy to filter misclassifications in the UI.

## Step 4 · Logging an Entire Directory with `mlflow.log_artifacts`

When a run produces many related files — feature engineering outputs, evaluation reports, generated data splits — you can log the entire directory in one call:

```python
mlflow.log_artifacts(local_dir="/tmp/my-outputs", artifact_path="outputs")
```

This **recursively copies** every file in `local_dir` to `artifact_path/` in the run's artifact store, preserving the directory structure.

Difference from `log_artifact`:

| Function | Input | Behavior |
|---|---|---|
| `log_artifact(path)` | Single file | Copies one file |
| `log_artifacts(dir)` | Directory | Recursively copies all files |
| `log_figure(fig, path)` | In-memory figure | No file needed |
| `log_table(df, file)` | DataFrame / dict | JSON stored in artifact store |

In [ ]:
import os
import json
import mlflow
import mlflow.sklearn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Build a local output directory to simulate a real pipeline ---
out_dir = "/tmp/iris-run-outputs"
os.makedirs(out_dir, exist_ok=True)
os.makedirs(f"{out_dir}/plots", exist_ok=True)
os.makedirs(f"{out_dir}/reports", exist_ok=True)

rf = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

# 1. Classification report as text
report = classification_report(
    y_test, y_pred, target_names=iris.target_names
)
with open(f"{out_dir}/reports/classification_report.txt", "w") as f:
    f.write(report)

# 2. Feature names list
with open(f"{out_dir}/reports/feature_names.txt", "w") as f:
    f.write("\n".join(iris.feature_names))

# 3. Run config as JSON
config = {"n_estimators": 50, "max_depth": 4, "random_state": 42, "test_accuracy": round(acc, 4)}
with open(f"{out_dir}/reports/config.json", "w") as f:
    json.dump(config, f, indent=2)

# 4. Feature importance plot
fig, ax = plt.subplots(figsize=(6, 3))
idx = np.argsort(rf.feature_importances_)[::-1]
ax.bar(range(4), rf.feature_importances_[idx], color="steelblue")
ax.set_xticks(range(4))
ax.set_xticklabels([iris.feature_names[i] for i in idx], rotation=30, ha="right")
ax.set_title("Feature Importances")
fig.tight_layout()
fig.savefig(f"{out_dir}/plots/feature_importance.png", dpi=100)
plt.close(fig)

# --- Log everything in one call ---
mlflow.set_experiment("iris-artifacts")

with mlflow.start_run(run_name="rf-log-artifacts-dir") as run:
    mlflow.log_params({"n_estimators": 50, "max_depth": 4})
    mlflow.log_metric("test_accuracy", acc)

    # Recursively upload all files under out_dir
    mlflow.log_artifacts(local_dir=out_dir, artifact_path="run-outputs")

    print(f"Run ID        : {run.info.run_id[:8]}")
    print(f"Test accuracy : {acc:.4f}")
    print(f"Uploaded directory: {out_dir}")
    print("Files logged:")
    for root, dirs, files in os.walk(out_dir):
        rel = os.path.relpath(root, out_dir)
        for fname in files:
            print(f"  run-outputs/{'' if rel == '.' else rel + '/'}{fname}")

### What just happened?
- **`log_artifacts(local_dir=out_dir, artifact_path="run-outputs")`** uploaded 4 files in 3 subdirectories with a single call.
- The directory tree is **preserved** inside `run-outputs/` — plots under `plots/`, reports under `reports/`.
- This pattern mirrors real ML pipelines: write all outputs to a local temp directory, then log the whole thing at the end of the run.
- In the artifact browser, you can click any `.txt` or `.json` file to preview it inline.

## Step 5 · Navigating the Artifact Browser Programmatically

After logging, you can verify what was stored without opening the UI. **`MlflowClient.list_artifacts`** returns the top-level entries for a run; pass an `artifact_path` to recurse into subdirectories.

To download an artifact back to disk:

```python
local_path = mlflow.artifacts.download_artifacts(
    artifact_uri="runs:/<run_id>/plots/confusion_matrix.png"
)
```

This is how you would retrieve a logged model, evaluation report, or feature list in a deployment or CI pipeline.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the most recent run from the iris-artifacts experiment
exp = mlflow.get_experiment_by_name("iris-artifacts")
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["attribute.start_time DESC"],
    max_results=1,
)
run_id = runs[0].info.run_id
print(f"Inspecting run: {run_id[:8]} ({runs[0].info.run_name})")

def walk_artifacts(run_id, path="", indent=0):
    """Recursively list all artifacts for a run, mirroring the UI browser."""
    entries = client.list_artifacts(run_id, path)
    for entry in entries:
        prefix = "  " * indent
        if entry.is_dir:
            print(f"{prefix}📁 {entry.path}/")
            walk_artifacts(run_id, entry.path, indent + 1)
        else:
            size_kb = entry.file_size / 1024 if entry.file_size else 0
            print(f"{prefix}📄 {entry.path}  ({size_kb:.1f} KB)")

print("\nArtifact tree:")
walk_artifacts(run_id)

### What just happened?
- **`client.list_artifacts(run_id, path)`** returns `FileInfo` objects with `.path`, `.is_dir`, and `.file_size`.
- The recursive `walk_artifacts` function mirrors the tree view in the UI's Artifacts browser.
- **`.file_size`** is in bytes; divide by 1024 for KB. Models are typically the largest artifacts.
- You can feed any `entry.path` back into `download_artifacts` to retrieve a specific file programmatically.

## Step 6 · Putting It All Together: A Self-Contained Run

A **self-contained run** logs everything needed to reproduce and understand the result:

| What | How |
|---|---|
| Hyperparameters | `log_params` |
| Evaluation metrics | `log_metrics` |
| Trained model | `mlflow.sklearn.log_model` |
| Evaluation plots | `log_figure` |
| Prediction table | `log_table` |
| Supporting files | `log_artifacts` |

The guiding principle: **someone with only the run ID and the tracking store should be able to reproduce your results and deploy your model** — no local files needed.

The cell below combines all techniques from today into a single production-style run.

In [ ]:
import os
import json
import mlflow
import mlflow.sklearn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
)

iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mlflow.set_experiment("iris-artifacts")

with mlflow.start_run(run_name="gbm-self-contained") as run:

    # 1. Params
    params = {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "random_state": 42}
    mlflow.log_params(params)
    mlflow.set_tags({"model_type": "gradient_boosting", "dataset": "iris"})

    # 2. Train
    gbm = GradientBoostingClassifier(**params)
    gbm.fit(X_train, y_train)
    y_pred  = gbm.predict(X_test)
    y_proba = gbm.predict_proba(X_test)

    # 3. Metrics
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="macro")
    mlflow.log_metrics({"test_accuracy": acc, "f1_macro": f1})

    # 4. Model
    mlflow.sklearn.log_model(gbm, artifact_path="model")

    # 5. Confusion matrix figure
    fig_cm, ax_cm = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_estimator(
        gbm, X_test, y_test, display_labels=iris.target_names, ax=ax_cm
    )
    ax_cm.set_title("GBM Confusion Matrix")
    fig_cm.tight_layout()
    mlflow.log_figure(fig_cm, "plots/confusion_matrix.png")
    plt.close(fig_cm)

    # 6. Predictions table
    pred_df = pd.DataFrame({
        "true_label": [iris.target_names[i] for i in y_test],
        "pred_label": [iris.target_names[i] for i in y_pred],
        "correct"   : y_test == y_pred,
    })
    mlflow.log_table(data=pred_df, artifact_file="tables/predictions.json")

    # 7. Supporting files as a directory
    supp_dir = "/tmp/gbm-support"
    os.makedirs(supp_dir, exist_ok=True)
    with open(f"{supp_dir}/classification_report.txt", "w") as fh:
        fh.write(classification_report(y_test, y_pred, target_names=iris.target_names))
    with open(f"{supp_dir}/feature_names.txt", "w") as fh:
        fh.write("\n".join(iris.feature_names))
    mlflow.log_artifacts(local_dir=supp_dir, artifact_path="support")

    print(f"Run ID    : {run.info.run_id[:8]}")
    print(f"Accuracy  : {acc:.4f}")
    print(f"F1 macro  : {f1:.4f}")
    print("Artifacts : model/, plots/, tables/, support/")

### What just happened?
- **All six artifact types** are logged in a single run — params, metrics, model, figure, table, and a directory.
- The run is **self-contained**: given the run ID, a colleague can reload the model, inspect the confusion matrix, and reproduce the predictions without any local files.
- **`GradientBoostingClassifier`** demonstrates that all these logging calls work identically for any sklearn estimator.
- In the MLflow UI, you will see four top-level artifact directories: `model/`, `plots/`, `tables/`, and `support/`.

In [ ]:
# Challenge: Complete self-contained run with a new estimator
#
# Instructions:
#   1. Train an SVC (sklearn.svm.SVC with probability=True) on the Iris dataset
#   2. Create a single MLflow run named "svc-full-logging" in the "iris-artifacts" experiment
#   3. Log all of the following:
#      a. Params: C, kernel, gamma
#      b. Metrics: test_accuracy and f1_macro
#      c. The trained model with mlflow.sklearn.log_model
#      d. A confusion matrix figure with mlflow.log_figure
#      e. A predictions DataFrame with mlflow.log_table
#      f. A text file containing classification_report as an artifact
#   4. Use walk_artifacts (defined above) to verify the artifact tree
#
# Scaffold:

# from sklearn.svm import SVC

# svc_params = {"C": ???, "kernel": "rbf", "gamma": "scale", "probability": True, "random_state": 42}

# mlflow.set_experiment("iris-artifacts")

# with mlflow.start_run(run_name="svc-full-logging") as run:
#     # ... your code here ...
#     pass

# walk_artifacts(run.info.run_id)

# Your solution here


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| `mlflow.sklearn.log_model` | Saves model + MLmodel manifest + conda/pip environment |
| `mlflow.sklearn.load_model(uri)` | Reload from `runs:/<id>/path` — no original object needed |
| `mlflow.log_figure(fig, path)` | Logs in-memory matplotlib figure — no temp file needed |
| `mlflow.log_table(df, file)` | DataFrame → JSON; interactive Tables tab in UI |
| `mlflow.log_artifact(path)` | Copy a single local file into the artifact store |
| `mlflow.log_artifacts(dir)` | Recursively copy an entire directory |
| `client.list_artifacts(run_id)` | Programmatic artifact browser — same data as the UI |
| Self-contained run | Log model + plots + tables + reports together |

> **Tip:** `log_artifact` copies a local file into the run's artifact store — always log your model, feature list, and evaluation plots together so a run is self-contained.

---
## What's next
**Day 4** → Register your best model in the MLflow Model Registry, transition it through Staging → Production, and load it using a registered model URI.

Mark Day 3 complete in your [tracker](../index.html).